In [0]:
%sql
-- Enable change data feed at existing table
ALTER TABLE dev.silver.customers_silver 
SET TBLPROPERTIES(delta.enableChangeDataFeed = TRUE)

In [0]:
%sql
DESC EXTENDED dev.silver.customers_silver

In [0]:
%sql
select * from dev.silver.customers_silver

In [0]:
# load new data to raw
#dbutils.fs.ls("s3://dalhussein-courses/DE-Pro/datasets/bookstore/v1/kafka-streaming/06.json")
dbutils.fs.cp("s3://dalhussein-courses/DE-Pro/datasets/bookstore/v1/kafka-streaming/10.json", "dbfs:/Volumes/dev/pro_landing_zone/kafka_sources/books_kafka_row/")

In [0]:
customer_schema = """
    customer_id STRING,
    email STRING,
    first_name STRING,
    last_name STRING,
    gender STRING,
    street STRING,
    city STRING,
    country_code STRING,
    row_status STRING,
    row_time TIMESTAMP
"""


In [0]:
# recreate functions for bronze and silver layer tables related to books data

# Process orders from bronze to silver
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def batch_upsert(microBatchDF, batchId):
    window = Window.partitionBy("customer_id").orderBy(F.col("row_time").desc())

    (
        microBatchDF.filter(F.col('row_status').isin(["insert", "update"]))
            .withColumn("rank", F.rank().over(window))
            .filter(F.col('rank') == 1)
            .drop("rank")
            .createOrReplaceTempView("ranked_customers")
    )

    sql_query = """
        MERGE INTO dev.silver.customers_silver c
        USING ranked_customers rc
        ON c.customer_id = rc.customer_id
        WHEN MATCHED AND c.row_time < rc.row_time THEN
            UPDATE SET *
        WHEN NOT MATCHED THEN
            INSERT *
    """
    microBatchDF.sparkSession.sql(sql_query)

def process_orders_silver():
    df = (
        spark.readStream
            .table("dev.multiplex_bronze.kafka_bronze")
            .filter(F.col("topic") == "orders")
            .select(
                F.from_json(
                    F.col("value").cast("string"),
                    "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>"
                ).alias("v")
            )
            .select("v.*")
            .filter(F.col("quantity") > 0)
            .writeStream
            .option("checkpointLocation", "/Volumes/dev/pro_landing_zone/checkpoints/silver_orders/")
            .trigger(availableNow=True)
            .table("dev.silver.orders_silver")
    )
    df.awaitTermination()

def process_bronze():
    df = (
      spark.readStream.format("cloudFiles")
          .option("cloudFiles.format", "json")
          .schema (schema="key BINARY, value BINARY, topic STRING, partition LONG, offset LONG, timestamp LONG")
          .option("pathGlobFilter", "*.json")
          .load("/Volumes/dev/pro_landing_zone/kafka_sources/books_kafka_row/")
          .withColumn("timestamp", (F.col("timestamp")/1000).cast("timestamp"))
          .withColumn("year_month", F.date_format("timestamp", "yyyy-MM"))
        .writeStream
          .option("checkpointLocation","/Volumes/dev/pro_landing_zone/checkpoints/bronze/")
          .option("mergeSchema", True)
          .partitionBy("topic", "year_month")
          .trigger(availableNow=True)
          .table("dev.multiplex_bronze.kafka_bronze")
    )
    df.awaitTermination()

df_country_lookup = spark.read.json("/Volumes/dev/pro_landing_zone/kafka_sources/books_kafka_row/coutry_lookup/*")
def process_customers_silver():
    (
        spark.readStream
                .table('dev.multiplex_bronze.kafka_bronze')
                .filter(F.col("topic") == "customers")
                .select(F.from_json(F.col("value").cast("string"), schema=customer_schema).alias('v'))
                .select('v.*')
                .join(
                    F.broadcast(df_country_lookup),
                    F.col("country_code") == F.col("code"),
                    "inner"
                )
            .writeStream
            .foreachBatch(batch_upsert)
            .option("checkpointLocation", "/Volumes/dev/pro_landing_zone/checkpoints/customers_silver")
            .trigger(availableNow=True)
            .start()

    )

In [0]:
process_bronze()
process_orders_silver()
process_customers_silver()

In [0]:
%sql
DESC HISTORY dev.silver.customers_silver

In [0]:
%sql
select * from dev.silver.customers_silver

In [0]:
%sql
select
  *
from table_changes("dev.silver.customers_silver", 2)

In [0]:
dbutils.fs.rm("/Volumes/dev/pro_landing_zone/checkpoints/customers_silver/display/", recurse=True) 
cdf_df = (
    spark.readStream
        .format("delta")
        .option("readChangeData", True)
        .option("startingVersion", 2)
        .table("dev.silver.customers_silver")
)
display(cdf_df, checkpointLocation = "/Volumes/dev/pro_landing_zone/checkpoints/customers_silver/display")